# Swiggy vs Zomato Sentiment Analysis
### Consumer Sentiment Breakdown & Keyword Targeting
This Jupyter Notebook performs end-to-end sentiment analysis on consumer commentary from the dataset `Swiggy vs Zomato.xlsx`.

#### Objectives:
1. Identify and isolate comments explicitly mentioning **'swiggy'**, **'zomato'**, or **both**.
2. Score sentiment polarity (Positive, Negative, Neutral) using a domain-tailored food-delivery lexicon.
3. Compute exact distribution counts and percentages.
4. Visualize comparative sentiment metrics with publication-quality charts.
5. Export clean JSON summaries.


In [ ]:
# Step 1: Import Required Libraries
import os
import re
import json
import pandas as pd
import matplotlib.pyplot as plt

# Display plots inline
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


In [ ]:
# Step 2: Locate and Load the Dataset
# Supports local folder, Downloads folder, and parent directory
possible_paths = [
    "Swiggy vs Zomato.xlsx",
    r"C:\Users\manka\Downloads\Swiggy vs Zomato.xlsx",
    os.path.join("..", "Swiggy vs Zomato.xlsx")
]

DATASET_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        DATASET_PATH = p
        break

if not DATASET_PATH:
    raise FileNotFoundError("Could not locate 'Swiggy vs Zomato.xlsx'. Please place it in the same folder.")

print(f"Loading dataset from: {DATASET_PATH}")
xl = pd.ExcelFile(DATASET_PATH)
df_swiggy = xl.parse("Swiggy")
df_zomato = xl.parse("zomato")

df_swiggy["brand_sheet"] = "Swiggy"
df_zomato["brand_sheet"] = "Zomato"

df_all = pd.concat([df_swiggy, df_zomato], ignore_index=True)
df_all = df_all.rename(columns={"Comment text": "comment_text"})
df_all["comment_text"] = df_all["comment_text"].astype(str).str.strip()

print(f"Loaded {len(df_swiggy)} Swiggy rows and {len(df_zomato)} Zomato rows. Total comments: {len(df_all)}")
df_all.head(5)


In [ ]:
# Step 3: Domain-Specific Sentiment Lexicon & Scoring Function
def has_word(text, word):
    """Checks whether a whole word exists in text (case-insensitive)."""
    if not isinstance(text, str):
        return False
    return bool(re.search(r"\b" + re.escape(word) + r"\b", text, re.IGNORECASE))

def analyze_comment_sentiment(text):
    """
    Computes sentiment polarity and label (Positive, Negative, Neutral)
    using domain-specific lexicons tailored for food delivery & quick commerce.
    """
    t = str(text).lower()
    
    pos_words = [
        "good", "great", "love", "loved", "saved", "discount", "lucky", "helpful",
        "refund received", "refunded", "resolved", "solution", "tasty", "delicious",
        "kudos", "respect", "awesome", "decent", "happy", "better", "best", "useful",
        "win", "free cash", "proper", "generous", "heartwarming", "polite", "cheaper"
    ]
    
    neg_words = [
        "scam", "scammed", "pathetic", "worst", "chutiya", "loot", "fraud", "stole",
        "penalty", "cancel", "cancellation", "harassment", "abused", "creepy", "threaten",
        "fight", "lawsuit", "complaint", "nch", "consumer court", "terrible", "stale",
        "cold", "soggy", "bug", "glitch", "fake", "dirty", "uninstalled", "deleted",
        "shitty", "disappointed", "underweight", "spilled", "hate", "robbery", "dumbest",
        "bad", "fuck", "bkl", "overpriced", "expensive", "rip off", "waste", "stuck"
    ]
    
    p = sum(1 for w in pos_words if w in t)
    n = sum(1 for w in neg_words if w in t)
    
    # Contextual high-severity friction weighting
    if any(k in t for k in ["charged 100%", "balance is negative", "not delivered", "never came", "stole my", "did nothing"]):
        n += 2
    if any(k in t for k in ["refund was processed", "initiated a refund", "got full refund", "always got good food"]):
        p += 2
        
    diff = p - n
    if diff > 0:
        return "Positive", round(min(1.0, 0.4 + diff * 0.15), 2)
    elif diff < 0:
        return "Negative", round(max(-1.0, -0.4 + diff * 0.15), 2)
    return "Neutral", 0.0


In [ ]:
# Step 4: Execute Sentiment Classification & Keyword Filtering
df_all["mentions_swiggy"] = df_all["comment_text"].apply(lambda t: has_word(t, "swiggy"))
df_all["mentions_zomato"] = df_all["comment_text"].apply(lambda t: has_word(t, "zomato"))

res = df_all["comment_text"].apply(analyze_comment_sentiment)
df_all["sentiment"] = [r[0] for r in res]
df_all["polarity_score"] = [r[1] for r in res]

def calculate_stats(df_subset, label):
    total = len(df_subset)
    counts = df_subset["sentiment"].value_counts().to_dict()
    pos = counts.get("Positive", 0)
    neg = counts.get("Negative", 0)
    neu = counts.get("Neutral", 0)
    return {
        "category": label,
        "total_comments": total,
        "negative_count": neg,
        "negative_pct": round((neg / max(1, total)) * 100, 2),
        "positive_count": pos,
        "positive_pct": round((pos / max(1, total)) * 100, 2),
        "neutral_count": neu,
        "neutral_pct": round((neu / max(1, total)) * 100, 2)
    }

swiggy_stats = calculate_stats(df_all[df_all["mentions_swiggy"]], "Mentions 'Swiggy'")
zomato_stats = calculate_stats(df_all[df_all["mentions_zomato"]], "Mentions 'Zomato'")
both_stats = calculate_stats(df_all[df_all["mentions_swiggy"] & df_all["mentions_zomato"]], "Mentions BOTH Brands")

summary_df = pd.DataFrame([swiggy_stats, zomato_stats, both_stats])
summary_df


In [ ]:
# Step 5: Visualizing Comparative Sentiment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Grouped Bar Chart
categories = summary_df["category"].tolist()
neg_pcts = summary_df["negative_pct"].tolist()
pos_pcts = summary_df["positive_pct"].tolist()
neu_pcts = summary_df["neutral_pct"].tolist()

x = range(len(categories))
width = 0.25

axes[0].bar([i - width for i in x], neg_pcts, width=width, label='Negative %', color='#C0392B')
axes[0].bar(x, pos_pcts, width=width, label='Positive %', color='#27AE60')
axes[0].bar([i + width for i in x], neu_pcts, width=width, label='Neutral %', color='#7F8C8D')

axes[0].set_ylabel('Percentage of Comments (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Sentiment Distribution Comparison', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories, fontsize=10)
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Subplot 2: Swiggy vs Zomato Direct Head-to-Head
pie_labels = ['Negative', 'Positive', 'Neutral']
colors = ['#C0392B', '#27AE60', '#BDC3C7']

swiggy_counts = [swiggy_stats["negative_count"], swiggy_stats["positive_count"], swiggy_stats["neutral_count"]]
zomato_counts = [zomato_stats["negative_count"], zomato_stats["positive_count"], zomato_stats["neutral_count"]]

axes[1].pie(swiggy_counts, labels=pie_labels, autopct='%1.1f%%', colors=colors, startangle=140)
axes[1].set_title(f"Swiggy Mentions Breakdown (N={swiggy_stats['total_comments']})", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Step 6: Export Results to JSON
output_json_path = "sentiment_analysis_results.json"
summary_dict = {
    "swiggy": swiggy_stats,
    "zomato": zomato_stats,
    "both": both_stats
}

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_dict, f, indent=2)

print(f"Results successfully saved to '{output_json_path}'!")
